# AC-OPF Relaxation Test — IEEE 9-bus

Solves a single AC-OPF instance at nominal demand using three methods and compares their optimal values:
- **Local solver** (pandapower / IPOPT) — true AC-OPF, non-convex
- **SOCP relaxation** (Jabr) — convex lower bound
- **SDP relaxation** (Lavaei-Low, real 2n×2n) — tightest convex lower bound

In [1]:
import pathlib, sys
PROJECT_ROOT = pathlib.Path(".").resolve().parents[0]
sys.path.insert(0, str(PROJECT_ROOT))

from problems.acopf.network import load_network
from problems.acopf.problem import solve_local, solve_relaxation

In [11]:
net, nd = load_network("case118")

print(f"Buses:       {nd.n_buses}")
print(f"Branches:    {len(nd.branch_from)}")
print(f"Generators:  {nd.n_gens}")
print(f"Loads:       {nd.n_loads}")
print(f"Base MVA:    {nd.baseMVA}")

Buses:       118
Branches:    179
Generators:  54
Loads:       99
Base MVA:    100.0


## Build the nominal demand vector

Use `α = 1`, `η = 1` everywhere — i.e. exactly the reference demand stored in the network.

In [12]:
import numpy as np

p_nominal = np.concatenate([nd.pd_nominal, nd.qd_nominal])  # [Pd (MW), Qd (MVar)]
print(f"Total nominal Pd: {nd.pd_nominal.sum():.1f} MW")
print(f"Total nominal Qd: {nd.qd_nominal.sum():.1f} MVar")

Total nominal Pd: 4242.0 MW
Total nominal Qd: 1438.0 MVar


## Local solver (pandapower / IPOPT)

In [13]:
local_val, local_res = solve_local(p_nominal, args={"net": net})
print(f"Local (AC-OPF):  {local_val:.4f}  [converged: {local_res['success']}]")

Local (AC-OPF):  129704.7402  [converged: True]


## SOCP relaxation (Jabr)

In [14]:
socp_val, socp_res = solve_relaxation(p_nominal, args={"nd": nd, "relaxation": "socp"})
print(f"SOCP relaxation: {socp_val:.4f}  [status: {socp_res['status']}, exact: {socp_res['exact']}]")

SOCP relaxation: 129383.6713  [status: optimal, exact: False]


## SDP relaxation (Lavaei-Low, real 2n×2n)

In [ ]:
sdp_val, sdp_res = solve_relaxation(p_nominal, args={"nd": nd, "relaxation": "sdp"})
print(f"SDP relaxation:  {sdp_val:.4f}  [status: {sdp_res['status']}, exact: {sdp_res['exact']}]")

## Comparison

In [ ]:
print(f"{'Method':<20} {'Value':>12} {'Gap vs local':>14}")
print("-" * 48)
for label, val in [("Local (AC-OPF)", local_val), ("SOCP", socp_val), ("SDP", sdp_val)]:
    gap = 100 * (local_val - val) / local_val if local_val else float("nan")
    print(f"{label:<20} {val:>12.4f} {gap:>13.4f}%")